In [1]:
import torch
import torch.nn as nn
from torchvision.transforms import ToTensor
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import RobustScaler

In [2]:
#load trained model
class AE(nn.Module):
    def __init__(self, input_size=103, latent_size=10):
        super(AE, self).__init__()
        self.input_size = input_size
        self.latent_size = latent_size

        # encoder network
        self.encoder = nn.Sequential(
            nn.Linear(input_size, 56), nn.ReLU(),
            nn.Linear(56, 28), nn.ReLU(),
            nn.Linear(28, latent_size)
        )
        # decoder network
        self.decoder = nn.Sequential(
            nn.Linear(latent_size, 28), nn.ReLU(),
            nn.Linear(28, 56), nn.ReLU(),
            nn.Linear(56, input_size)
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        z = self.encode(x)
        return self.decode(z)

def load_model(input_size, latent_size):
    model = AE(input_size=input_size, latent_size=latent_size)
    model.load_state_dict(
        torch.load("/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/autoencoder_lr_scheduler.pth")
    )
    model.eval()
    return model
model = load_model(103, 10)

In [8]:
path = "/home/preclineu/quirom/Desktop/internship/Normative-Models-and-VAE/logs/Autoencoder/unseen_data/unseen_data.csv"

df_unseen = pd.read_csv(path)
df_unseen = df_unseen.iloc[:,1:]

scaler = RobustScaler()

data_scaled = scaler.fit_transform(df_unseen)
data_tensor = torch.tensor(data_scaled, dtype=torch.float32)
data_tensor = TensorDataset(data_tensor)
data_loader = DataLoader(data_tensor, batch_size=64, shuffle=True)


In [10]:
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def evaluate_reconstruction(model, dataloader, device):
    model.eval()
    all_outputs = []
    all_inputs = []

    with torch.no_grad():
        for batch in dataloader:
            x, = batch
            x = x.to(device)
            output = model(x)
            all_outputs.append(output.cpu().numpy())
            all_inputs.append(x.cpu().numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    all_inputs = np.concatenate(all_inputs, axis=0)

    mse = mean_squared_error(all_inputs, all_outputs)
    mae = mean_absolute_error(all_inputs, all_outputs)
    r2 = r2_score(all_inputs, all_outputs)

    # print(f"MSE: {mse:.4f}")
    # print(f"MAE: {mae:.4f}")
    # print(f"R² Score: {r2:.4f}")

    return mse, mae, r2

mse, mae, r2 = evaluate_reconstruction(model, data_loader, device)
print("performance on unseen data:")
print(f"MSE: {mse:.4f}")
print(f"MAE: {mae:.4f}")
print(f"R² Score: {r2:.4f}")



performance on unseen data:
MSE: 0.2182
MAE: 0.3594
R² Score: 0.6417


performance on test data (that model used to finetune):
MSE: 0.2175
MAE: 0.3590
R² Score: 0.6436